# 📊 Notebook 01 — Análisis Exploratorio de Datos (EDA)
## EnergyForecast · IA EAFIT 2026-1
**Dataset:** PJM Hourly Energy Consumption  
**Fuente:** https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption

### Instrucciones:
1. Descarga el dataset de Kaggle (archivo `PJME_hourly.csv`)
2. Súbelo a Colab o colócalo en `data/raw/PJME_hourly.csv`
3. Ejecuta las celdas en orden

In [ ]:
# ── Instalación de dependencias (solo en Colab) ──────────────────────────────
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn', 'statsmodels']:
    install(pkg)
print('✅ Dependencias instaladas')

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficas
plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
EAFIT_BLUE = '#003d79'
EAFIT_GREEN = '#009650'
print('✅ Imports OK')

## 1. Carga y descripción del dataset

In [ ]:
# ── Carga del dataset ────────────────────────────────────────────────────────
# Si estás en Colab, sube el archivo primero con:
# from google.colab import files; files.upload()

DATA_PATH = 'data/raw/PJME_hourly.csv'   # ajusta si es necesario

try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    # Intento alternativo (nombre directo si se subió a Colab)
    df = pd.read_csv('PJME_hourly.csv')

# Parseo de fechas e índice temporal
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.set_index('Datetime').sort_index()
df.columns = ['MW']   # renombrar columna de consumo

print(f'Shape: {df.shape}')
print(f'Rango temporal: {df.index.min()} → {df.index.max()}')
print(f'Años cubiertos: {df.index.year.nunique()} ({df.index.year.min()}–{df.index.year.max()})')
df.head(10)

In [ ]:
# ── Estadísticas descriptivas ────────────────────────────────────────────────
desc = df.describe()
print('\n📋 Estadísticas descriptivas:')
print(desc.round(2))

# Valores nulos
nulls = df.isnull().sum()
print(f'\n🔍 Valores nulos: {nulls.values[0]}')

# Duplicados
dups = df.index.duplicated().sum()
print(f'🔍 Timestamps duplicados: {dups}')

## 2. Visualización de la serie temporal

In [ ]:
# ── Serie temporal completa ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle('PJM East — Consumo Eléctrico Horario', fontsize=14, fontweight='bold', color=EAFIT_BLUE)

# Serie completa
axes[0].plot(df.index, df['MW'], color=EAFIT_BLUE, linewidth=0.4, alpha=0.8)
axes[0].set_title('Serie completa (2002–2018)', fontsize=11)
axes[0].set_ylabel('MW')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Zoom: un año
df_2015 = df['2015']
axes[1].plot(df_2015.index, df_2015['MW'], color=EAFIT_GREEN, linewidth=0.6)
axes[1].set_title('Zoom: año 2015', fontsize=11)
axes[1].set_ylabel('MW')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Zoom: una semana
df_week = df['2015-01-05':'2015-01-11']
axes[2].plot(df_week.index, df_week['MW'], color='#e05c00', linewidth=1.5, marker='o', markersize=2)
axes[2].set_title('Zoom: semana típica (5–11 ene 2015)', fontsize=11)
axes[2].set_ylabel('MW')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%a %d'))

plt.tight_layout()
plt.savefig('data/processed/fig01_serie_temporal.png', bbox_inches='tight')
plt.show()
print('💾 Figura guardada: fig01_serie_temporal.png')

## 3. Análisis de distribución y estacionalidad

In [ ]:
# ── Ingeniería de características temporales ─────────────────────────────────
df['hour']       = df.index.hour
df['dayofweek']  = df.index.dayofweek   # 0=lunes, 6=domingo
df['month']      = df.index.month
df['quarter']    = df.index.quarter
df['year']       = df.index.year
df['dayofyear']  = df.index.dayofyear
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

print('✅ Features temporales creadas:', [c for c in df.columns if c != 'MW'])

In [ ]:
# ── Patrones de estacionalidad ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Patrones de Estacionalidad — PJM Energy', fontsize=13, fontweight='bold', color=EAFIT_BLUE)

# Por hora del día
hourly = df.groupby('hour')['MW'].mean()
axes[0,0].plot(hourly.index, hourly.values, color=EAFIT_BLUE, linewidth=2, marker='o', markersize=4)
axes[0,0].fill_between(hourly.index, hourly.values, alpha=0.15, color=EAFIT_BLUE)
axes[0,0].set_title('Promedio por hora del día', fontweight='bold')
axes[0,0].set_xlabel('Hora')
axes[0,0].set_ylabel('MW promedio')
axes[0,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Por día de la semana
day_names = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']
weekly = df.groupby('dayofweek')['MW'].mean()
colors = [EAFIT_BLUE]*5 + ['#e05c00']*2
axes[0,1].bar(day_names, weekly.values, color=colors, edgecolor='white', linewidth=0.5)
axes[0,1].set_title('Promedio por día de la semana', fontweight='bold')
axes[0,1].set_ylabel('MW promedio')
axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Por mes
month_names = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
monthly = df.groupby('month')['MW'].mean()
axes[1,0].bar(month_names, monthly.values, color=EAFIT_GREEN, edgecolor='white', linewidth=0.5)
axes[1,0].set_title('Promedio por mes', fontweight='bold')
axes[1,0].set_ylabel('MW promedio')
axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Boxplot por hora (distribución completa)
df_sample = df[df['year'].isin([2013,2014,2015])]
hour_data = [df_sample[df_sample['hour']==h]['MW'].values for h in range(24)]
bp = axes[1,1].boxplot(hour_data, patch_artist=True, showfliers=False,
                        medianprops=dict(color='white', linewidth=2))
for patch in bp['boxes']:
    patch.set_facecolor(EAFIT_BLUE)
    patch.set_alpha(0.7)
axes[1,1].set_title('Distribución MW por hora (2013–2015)', fontweight='bold')
axes[1,1].set_xlabel('Hora del día')
axes[1,1].set_ylabel('MW')
axes[1,1].set_xticks(range(1,25,2))
axes[1,1].set_xticklabels(range(0,24,2))
axes[1,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.savefig('data/processed/fig02_estacionalidad.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Distribución de la variable objetivo ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Distribución de la Variable Objetivo (MW)', fontsize=12, fontweight='bold', color=EAFIT_BLUE)

axes[0].hist(df['MW'], bins=80, color=EAFIT_BLUE, edgecolor='white', linewidth=0.3, alpha=0.85)
axes[0].set_title('Histograma de frecuencias')
axes[0].set_xlabel('MW')
axes[0].set_ylabel('Frecuencia')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

# Tendencia anual
yearly = df.groupby('year')['MW'].mean()
axes[1].plot(yearly.index, yearly.values, color=EAFIT_GREEN, linewidth=2, marker='s', markersize=6)
axes[1].fill_between(yearly.index, yearly.values, alpha=0.15, color=EAFIT_GREEN)
axes[1].set_title('Tendencia anual (media por año)')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('MW promedio')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.savefig('data/processed/fig03_distribucion.png', bbox_inches='tight')
plt.show()

print(f'Media: {df["MW"].mean():,.0f} MW')
print(f'Mediana: {df["MW"].median():,.0f} MW')
print(f'Desv. estándar: {df["MW"].std():,.0f} MW')
print(f'Mín: {df["MW"].min():,.0f} MW | Máx: {df["MW"].max():,.0f} MW')

## 4. Correlaciones y lag features

In [ ]:
# ── Correlación de features temporales con MW ─────────────────────────────────
corr_features = ['hour', 'dayofweek', 'month', 'quarter', 'is_weekend']
corr_vals = df[corr_features + ['MW']].corr()['MW'].drop('MW').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
colors = [EAFIT_BLUE if v > 0 else '#c0392b' for v in corr_vals.values]
bars = ax.barh(corr_vals.index, corr_vals.values, color=colors, edgecolor='white')
ax.set_title('Correlación de features temporales con consumo (MW)', fontweight='bold', color=EAFIT_BLUE)
ax.set_xlabel('Correlación de Pearson')
ax.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, corr_vals.values):
    ax.text(val + 0.002*np.sign(val), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val > 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig('data/processed/fig04_correlaciones.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── ACF y PACF ───────────────────────────────────────────────────────────────
# Usamos muestra para no saturar el plot (primeras 2000 obs)
df_acf = df['MW'].iloc[:2000]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Autocorrelación de la serie temporal', fontsize=12, fontweight='bold', color=EAFIT_BLUE)
plot_acf(df_acf, lags=168, ax=axes[0], color=EAFIT_BLUE, title='ACF (lags hasta 168h = 1 semana)')
plot_pacf(df_acf, lags=48, ax=axes[1], color=EAFIT_GREEN, title='PACF (lags hasta 48h)')
plt.tight_layout()
plt.savefig('data/processed/fig05_acf_pacf.png', bbox_inches='tight')
plt.show()

print('\n📝 Interpretación ACF:')
print('  - Pico fuerte en lag 24 → estacionalidad diaria')
print('  - Pico fuerte en lag 168 → estacionalidad semanal')
print('  - Decaimiento lento → serie con tendencia (no estacionaria)')

In [ ]:
# ── Test de estacionariedad (ADF) ─────────────────────────────────────────────
adf_result = adfuller(df['MW'].dropna(), autolag='AIC')
print('\n🧪 Test Augmented Dickey-Fuller:')
print(f'  Estadístico ADF: {adf_result[0]:.4f}')
print(f'  p-valor:         {adf_result[1]:.6f}')
print(f'  Valores críticos:')
for k, v in adf_result[4].items():
    print(f'    {k}: {v:.4f}')
if adf_result[1] < 0.05:
    print('\n✅ La serie ES estacionaria (p < 0.05)')
else:
    print('\n⚠️  La serie NO es estacionaria (p >= 0.05) → se necesita diferenciación')

## 5. Detección de anomalías y valores atípicos

In [ ]:
# ── Detección de outliers con IQR ─────────────────────────────────────────────
Q1 = df['MW'].quantile(0.25)
Q3 = df['MW'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 3 * IQR
upper = Q3 + 3 * IQR

outliers = df[(df['MW'] < lower) | (df['MW'] > upper)]
print(f'Límite inferior (Q1 - 3·IQR): {lower:,.0f} MW')
print(f'Límite superior (Q3 + 3·IQR): {upper:,.0f} MW')
print(f'Outliers detectados: {len(outliers)} ({len(outliers)/len(df)*100:.2f}% del dataset)')

if len(outliers) > 0:
    print('\nEjemplos de outliers:')
    print(outliers.head(10))

## 6. Preprocesamiento final y guardado

In [ ]:
# ── Pipeline de preprocesamiento ──────────────────────────────────────────────
print('🔧 Iniciando preprocesamiento...')

# 1. Rellenar huecos en el índice temporal (asegurar continuidad horaria)
full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='h')
df_clean = df[['MW']].reindex(full_idx)
gaps = df_clean['MW'].isnull().sum()
print(f'   Huecos temporales detectados: {gaps}')

# 2. Imputación con interpolación lineal
df_clean['MW'] = df_clean['MW'].interpolate(method='time')
df_clean.index.name = 'Datetime'
print(f'   Valores nulos tras imputación: {df_clean["MW"].isnull().sum()}')

# 3. Re-agregar features temporales
df_clean['hour']       = df_clean.index.hour
df_clean['dayofweek']  = df_clean.index.dayofweek
df_clean['month']      = df_clean.index.month
df_clean['quarter']    = df_clean.index.quarter
df_clean['year']       = df_clean.index.year
df_clean['dayofyear']  = df_clean.index.dayofyear
df_clean['is_weekend'] = (df_clean['dayofweek'] >= 5).astype(int)

# 4. Split temporal (train/val/test)
# Train: 2002–2014 | Val: 2015 | Test: 2016–2018
df_train = df_clean[:'2014']
df_val   = df_clean['2015':'2015']
df_test  = df_clean['2016':]

print(f'\n📦 Split temporal:')
print(f'   Train: {len(df_train):,} registros ({df_train.index.min().date()} → {df_train.index.max().date()})')
print(f'   Val:   {len(df_val):,} registros ({df_val.index.min().date()} → {df_val.index.max().date()})')
print(f'   Test:  {len(df_test):,} registros ({df_test.index.min().date()} → {df_test.index.max().date()})')
print(f'\n   Proporción: {len(df_train)/len(df_clean)*100:.1f}% / {len(df_val)/len(df_clean)*100:.1f}% / {len(df_test)/len(df_clean)*100:.1f}%')

# 5. Guardar
import os
os.makedirs('data/processed', exist_ok=True)
df_clean.to_csv('data/processed/PJME_clean.csv')
df_train.to_csv('data/processed/train.csv')
df_val.to_csv('data/processed/val.csv')
df_test.to_csv('data/processed/test.csv')
print('\n✅ Archivos guardados en data/processed/')

In [ ]:
# ── Resumen final del EDA ──────────────────────────────────────────────────────
print('='*55)
print('  RESUMEN DEL EDA — PJM Energy Consumption')
print('='*55)
print(f'  Registros totales:  {len(df_clean):,}')
print(f'  Rango temporal:     {df_clean.index.min().year}–{df_clean.index.max().year}')
print(f'  Variable objetivo:  MW (continua, regresión)')
print(f'  Media:              {df_clean["MW"].mean():,.0f} MW')
print(f'  Rango:              {df_clean["MW"].min():,.0f} – {df_clean["MW"].max():,.0f} MW')
print(f'  Outliers (3·IQR):  {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)')
print(f'  Huecos imputados:   {gaps}')
print(f'  Features creadas:   hour, dayofweek, month, quarter,')
print(f'                      year, dayofyear, is_weekend')
print('='*55)
print('\n✅ EDA completo. Continúa con 02_preprocessing.ipynb')